# prepare_phase3_gpu
Phase 3 L4 GPU Track Stage B1 (`phase3-gpu-stageB1.v1`): freeze deterministic `E23` disjoint opposite-label `SCREEN` image pairs and `E39+E31` 8-slice sub-caption + visual-token/attention extraction plan (`E35` dropped to keep G9-Q6 unified), and publish immutable `phase3_gpu_plan`.

- Run on Colab CPU (`requirements-stage2-cpu.txt`, Python 3.11/3.12, single BLAS/OpenMP thread); zero raw image IO.
- Edit only Cell 3 if `PHASE1_ROOT` or `CPU_PLAN_REF` differs.


In [ ]:
import os, sys, subprocess
from pathlib import Path

if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 with the pinned CPU requirements.')
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'

from google.colab import drive
drive.mount('/content/drive')

REPO = Path('/content/Research-phase3-gpu-prep')
CODE_REF = 'phase3'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Drazr/Research.git', str(REPO)], check=True)
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local work before continuing.')
subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO, check=True)
CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], cwd=REPO, text=True).strip()
subprocess.run(['git', 'checkout', '--detach', CODE_COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-stage2-cpu.txt')], check=True)
print('Fixed code commit:', CODE_COMMIT)
print('Restart if installation requests it, then rerun setup before continuing.')


In [ ]:
# Visible Stage B1 settings.
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
CPU_PLAN_REF = {
    'path': 'phase3/plans/20261120T020039Z-0d82ded3b7c3497e999d70a33f35c3a8/_COMPLETE.json',
    'sha256': '93cbf480e9d8c13401aaa98315444f3a4d2b3b57ec769b458a799dc269c40439',
}


In [ ]:
import json

if subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() != CODE_COMMIT:
    raise RuntimeError('Checkout changed after setup.')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean Phase 3 checkout.')
if not PHASE1_ROOT.is_dir():
    raise RuntimeError(f'Missing mounted PHASE1_ROOT directory: {PHASE1_ROOT}')

cmd = [
    sys.executable, '-u', '-m', 'src.phase3.gpu_plan', 'prepare',
    '--phase1-root', str(PHASE1_ROOT),
    '--storage-scope', 'persistent-drive',
]
if CPU_PLAN_REF is not None:
    cmd.extend(['--cpu-plan', json.dumps(CPU_PLAN_REF) if isinstance(CPU_PLAN_REF, dict) else str(CPU_PLAN_REF)])

proc = subprocess.Popen(cmd, cwd=REPO, env=os.environ.copy(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
blockers = []
last_complete = None
for line in proc.stdout:
    print(line, end='', flush=True)
    try:
        payload = json.loads(line)
        if isinstance(payload, dict):
            if payload.get('status') == 'blocked':
                blockers.extend(payload.get('blockers', []))
            if payload.get('event') == 'complete':
                last_complete = payload
    except json.JSONDecodeError:
        pass
code = proc.wait()
if code != 0 or last_complete is None:
    detail = '; '.join(blockers) if blockers else f'exit code {code}'
    raise RuntimeError(f'Phase 3 Stage B1 failed ({code}): {detail}')

print('\n=== AUTHORITATIVE PHASE 3 GPU PLAN REFERENCE ===')
print(json.dumps(last_complete['plan_ref'], indent=2))
print('REMINDER: Run drive.flush_and_unmount() before launching L4 GPU runtimes.')
